## Get Physical Properties from AlphaFold Output

In [2]:
# Install packages
%pip install -q biopython rdkit MDAnalysis prolif

Note: you may need to restart the kernel to use updated packages.


## Setup

In [1]:
import prolif as plf
import MDAnalysis as mda
from MDAnalysis.analysis import contacts
from rdkit import Chem
from rdkit.Chem import AllChem
import os
import numpy as np

C:\Users\ryangustafson\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\MDAnalysis\topology\tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)


## ProLIF

--- Setup ---
ProLIF works best by loading the protein from a PDB and the ligand from an SDF or MOL2 file, as this preserves bond order and chemistry.

Let's assume you have `my_complex.pdb`.
We first need to "split" it into protein and ligand.
This is a common preparatory step.

For this example, let's assume you already have:
1. 'protein.pdb' (just the protein chains)
2. 'ligand.mol' (just the ligand, with correct chemistry)

In [ ]:
import MDAnalysis as mda
import prolif as plf
from rdkit import Chem
from rdkit.Chem import AllChem
import warnings
import os

# Suppress common RDKit warnings
from rdkit import rdBase
rdBase.DisableLog('rdApp.warning')
warnings.filterwarnings('ignore', category=UserWarning)

# --- 1. Setup ---
complex_pdb = "my_complex.pdb" # <-- Your PDB file here
ligand_resname = "LIG"        # <-- Your ligand's residue name

# --- 2. Load and Select with MDAnalysis ---
try:
    # You were 100% correct to do this:
    u = mda.Universe(complex_pdb)
except Exception as e:
    print(f"Error loading {complex_pdb} with MDAnalysis: {e}")
    exit()

# Define the two parts of your complex
ligand_selection = f"resname {ligand_resname}"
protein_selection = f"protein and not resname {ligand_resname}"

lig_ag = u.select_atoms(ligand_selection)
prot_ag = u.select_atoms(protein_selection)

if lig_ag.n_atoms == 0:
    print(f"Error: No atoms found for ligand 'resname {ligand_resname}'.")
    print("Please check the resname in your PDB file.")
    exit()

# --- 3. Create ProLIF Molecules (The Fix) ---

# We need to create two 'prolif.Molecule' objects.

# --- For the PROTEIN: ---
# We can use the error's suggestion, as we don't need
# detailed chemistry for the "environment".
try:
    prot = plf.Molecule.from_mda(prot_ag, inferrer=None)
except Exception as e:
    # This might fail if the PDB is very unusual
    print(f"Error loading protein: {e}")
    print("Trying to load from a temporary PDB file...")
    prot_ag.write("temp_protein.pdb")
    prot_mol = Chem.MolFromPDBFile("temp_protein.pdb", removeHs=False)
    prot = plf.Molecule(prot_mol)
    os.remove("temp_protein.pdb")
    

# --- For the LIGAND: ---
# This is the critical step. We MUST add hydrogens.
# We will use RDKit to do this.

# A) Save the ligand atoms to a temporary PDB file
lig_ag.write("temp_ligand.pdb")

# B) Load the PDB file with RDKit
rdk_mol = Chem.MolFromPDBFile("temp_ligand.pdb", removeHs=False)

# C) "Sanitize" it: add hydrogens and infer bond orders
if rdk_mol is None:
    print("Error: RDKit could not read temp_ligand.pdb.")
    print("Your ligand structure may be non-standard.")
    exit()

try:
    # This is the magic step.
    rdk_mol_h = Chem.AddHs(rdk_mol, addCoords=True)
    
    # Assign bond orders from 3D coordinates
    # This is necessary since PDBs don't have them
    Chem.AssignBondOrdersFromTemplate(rdk_mol_h, rdk_mol)
    
except Exception:
    # Fallback if template-based bond order assignment fails
    # This is a less reliable method but better than nothing
    rdk_mol = Chem.MolFromPDBFile("temp_ligand.pdb", removeHs=False, proximityBonding=True)
    rdk_mol_h = Chem.AddHs(rdk_mol, addCoords=True)


# D) Create the final ProLIF ligand molecule
lig = plf.Molecule(rdk_mol_h)
    
# Clean up temporary file
os.remove("temp_ligand.pdb")

print("Successfully loaded and sanitized Protein and Ligand.")

# --- 4. Run Analysis ---
fp = plf.Fingerprint()

# Use 'run_from_molecules' now that we have them
fp.run_from_molecules(lig, prot)

# --- 5. Get Results ---
if not fp.ifp:
    print("\nAnalysis complete, but no interactions were found.")
else:
    # As a dictionary
    interaction_dict = fp.to_dict()
    print("\n--- ProLIF Interaction Dictionary ---")
    print(interaction_dict)

    # As a more readable Pandas DataFrame
    # (requires 'pip install pandas')
    try:
        df = fp.to_dataframe()
        print("\n--- ProLIF Interaction DataFrame ---")
        print(df)
    except ImportError:
        print("\nInstall 'pandas' (pip install pandas) to see the DataFrame summary.")

Error: No atoms found for ligand 'resname LIG'.
Please check the resname in your PDB file.


IndexError: Cannot write an AtomGroup with 0 atoms

: 

## MDAnalysis

In [7]:
import MDAnalysis as mda
from MDAnalysis.lib.distances import distance_array
import numpy as np

# Load your PDB file
pdb_file = "my_complex.pdb" # <-- Your PDB file here

try:
    u = mda.Universe(pdb_file)
except Exception as e:
    print(f"Error loading {pdb_file}: {e}")
    exit()

# --- 1. Define Your Specific Parts ---
sel_A = "chainID A"
sel_B = "chainID B"

group_A = u.select_atoms(sel_A)
group_B = u.select_atoms(sel_B)

if group_A.n_atoms == 0 or group_B.n_atoms == 0:
    print("Error: One or both selections resulted in 0 atoms.")
    print(f"Check if '{sel_A}' and '{sel_B}' are correct for your PDB.")
    exit()

print(f"Group A ({sel_A}): {group_A.n_atoms} atoms")
print(f"Group B ({sel_B}): {group_B.n_atoms} atoms")

# --- 2. Run Analysis (New Method) ---
distance_cutoff = 4.5

# Calculate the distance matrix between all atoms in group A
# and all atoms in group B.
# This creates a (n_atoms_A, n_atoms_B) array.
dist_matrix = distance_array(group_A.positions, group_B.positions)

# Find the indices (i, j) where the distance is less than the cutoff
# i = index in group_A, j = index in group_B
close_atom_indices = np.where(dist_matrix <= distance_cutoff)

# 'close_atom_indices' is a tuple of two arrays:
# (array_of_i_indices, array_of_j_indices)

# --- 3. Map Atom Indices to Residues ---

# Get the indices of the atoms in group_A that are close
close_atoms_A_indices = close_atom_indices[0]
# Get the indices of the atoms in group_B that are close
close_atoms_B_indices = close_atom_indices[1]

# Now, map these atom indices to their parent residues
# We use a set to store the residues so we only get unique ones
interacting_residues_A = set()
for atom_index in close_atoms_A_indices:
    interacting_residues_A.add(group_A[atom_index].residue)

interacting_residues_B = set()
for atom_index in close_atoms_B_indices:
    interacting_residues_B.add(group_B[atom_index].residue)

# --- 4. Print Results ---
print("\n--- MDAnalysis Contact Analysis (within 4.5 Å) ---")

print(f"\nResidues in Chain A interacting with Chain B:")
if interacting_residues_A:
    # Sort the residues by resid for a clean output
    sorted_residues = sorted(list(interacting_residues_A), key=lambda r: r.resid)
    print([res.resname + str(res.resid) for res in sorted_residues])
else:
    print("None found.")

print(f"\nResidues in Chain B interacting with Chain A:")
if interacting_residues_B:
    sorted_residues = sorted(list(interacting_residues_B), key=lambda r: r.resid)
    print([res.resname + str(res.resid) for res in sorted_residues])
else:
    print("None found.")

# --- Optional: Hydrogen Bond Analysis ---
# (This still has the same limitation: it needs hydrogens in the PDB)
try:
    from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
    
    # Select all protein atoms for H-bond analysis
    # We select 'protein' and 'protein' to find intra-protein H-bonds
    h = HydrogenBondAnalysis(u, sel1="protein", sel2="protein")
    h.run()
    
    # Filter for H-bonds specifically between Chain A and Chain B
    inter_chain_hbonds = []
    for bond in h.results.hbonds:
        donor_chain = u.atoms[bond[2]].chainID
        acceptor_chain = u.atoms[bond[0]].chainID
        
        if (donor_chain == 'A' and acceptor_chain == 'B') or \
           (donor_chain == 'B' and acceptor_chain == 'A'):
            inter_chain_hbonds.append(bond)

    print(f"\nFound {len(inter_chain_hbonds)} total inter-chain H-bonds (A-B).")
    # Note: This will likely be 0 if your PDB has no hydrogens.

except ImportError:
    print("\nHydrogen bond analysis requires an updated version of MDAnalysis.")
except Exception as e:
    print(f"\nCould not run H-bond analysis: {e}")

Group A (chainID A): 1520 atoms
Group B (chainID B): 1299 atoms

--- MDAnalysis Contact Analysis (within 4.5 Å) ---

Residues in Chain A interacting with Chain B:
['CYS1', 'THR2', 'CYS3', 'SER4', 'PRO5', 'PRO33', 'PHE34', 'GLY35', 'GLU62', 'SER64', 'GLU65', 'SER66', 'LEU67', 'CYS68', 'LYS71', 'ARG84', 'LEU94', 'CYS95', 'LYS123', 'LYS125', 'TYR128', 'TYR129', 'ASN148']

Residues in Chain B interacting with Chain A:
['PHE4', 'TYR73', 'ASP76', 'ASP79', 'GLY80', 'LEU81', 'LEU82', 'ALA83', 'HIS84', 'ALA85', 'PHE86', 'PRO87', 'ILE92', 'GLN93', 'GLY107', 'LYS108', 'GLN110', 'TYR112', 'VAL117', 'HIS120', 'GLU121', 'HIS124', 'ASP129', 'HIS130', 'PRO140', 'MET141', 'TYR142', 'ARG143', 'PHE144', 'GLU146']

Could not run H-bond analysis: HydrogenBondAnalysis.__init__() got an unexpected keyword argument 'sel1'. Did you mean 'self'?
